<a href="https://colab.research.google.com/github/Nayyab-Khalid/Apple-disease-detection/blob/main/Copy_of_CNNVGGResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report
import pandas as pd
import os


In [ ]:
# Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# Set dataset path
dataset_path = '/content/drive/MyDrive/vegetable/'
train_dir = os.path.join(dataset_path, 'train')
test_dir = os.path.join(dataset_path, 'test')
val_dir = os.path.join(dataset_path, 'validation')



In [ ]:
# Data Augmentation and Preprocessing
data_gen = ImageDataGenerator(rescale=1.0/255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)
train_data = data_gen.flow_from_directory(train_dir, target_size=(150, 150), batch_size=32, class_mode='categorical')
test_data = data_gen.flow_from_directory(test_dir, target_size=(150, 150), batch_size=32, class_mode='categorical', shuffle=False)
val_data = data_gen.flow_from_directory(val_dir, target_size=(150, 150), batch_size=32, class_mode='categorical')


Found 8000 images belonging to 8 classes.
Found 1600 images belonging to 8 classes.
Found 1610 images belonging to 8 classes.


In [ ]:
# 1. CNN Model
def build_cnn(num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
num_classes = len(train_data.class_indices)  # Number of classes
cnn_model = build_cnn(num_classes)
cnn_model.fit(train_data, validation_data=val_data, epochs=10)



/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


250/250 ━━━━━━━━━━━━━━━━━━━━ 1899s 7s/step - accuracy: 0.4058 - loss: 1.8100 - val_accuracy: 0.7820 - val_loss: 0.6247
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 504s 2s/step - accuracy: 0.7319 - loss: 0.7501 - val_accuracy: 0.8646 - val_loss: 0.4126
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 502s 2s/step - accuracy: 0.8044 - loss: 0.5552 - val_accuracy: 0.8988 - val_loss: 0.3143
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 467s 2s/step - accuracy: 0.8514 - loss: 0.4201 - val_accuracy: 0.9012 - val_loss: 0.2795
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 455s 2s/step - accuracy: 0.8680 - loss: 0.3808 - val_accuracy: 0.9106 - val_loss: 0.2543
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 494s 2s/step - accuracy: 0.8670 - loss: 0.3808 - val_accuracy: 0.9255 - val_loss: 0.2423
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 449s 2s/step - accuracy: 0.8864 - loss: 0.3158 - val_accuracy: 0.9478 - val_loss: 0.1813
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 442s 2s/step - accuracy: 0.9113 - loss: 0.2805 - val_accuracy: 0.95

In [ ]:
# 2. ResNet Model
def build_resnet(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
    base_model.trainable = False
    model = Sequential([
        base_model,
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
num_classes = len(train_data.class_indices)  # Define num_classes in this cell
resnet_model = build_resnet(num_classes)
resnet_model.fit(train_data, validation_data=val_data, epochs=10)



Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1012s 4s/step - accuracy: 0.1200 - loss: 2.3233 - val_accuracy: 0.1242 - val_loss: 2.0795
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 986s 4s/step - accuracy: 0.1249 - loss: 2.0795 - val_accuracy: 0.1242 - val_loss: 2.0795
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1042s 4s/step - accuracy: 0.1206 - loss: 2.0796 - val_accuracy: 0.1242 - val_loss: 2.0794
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1044s 4s/step - accuracy: 0.1213 - loss: 2.0795 - val_accuracy: 0.1242 - val_loss: 2.0795
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 991s 4s/step - accuracy: 0.1239 - loss: 2.0796 - val_accuracy: 0.1242 - val_loss: 2.0795
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1034s 4s/step - accuracy: 0.1150 - loss: 2.0796 - val_accuracy: 0.1242 - val_loss: 2.0795
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 992s 4s/step - accuracy: 0.1245 - loss: 2.0795 - val_accuracy: 0.1242 - val_loss: 2.0794
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1039s 4s/step - accuracy: 0.1285 - loss: 2.0795 - val